<a href="https://colab.research.google.com/github/jdospina/jump-diffusion-estimation/blob/main/notebooks/sp500_jump_diffusion_example_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 S&P 500 analysis with a jump-diffusion model

This notebook is a complete tutorial on applying a **jump-diffusion model** to analyze the returns of the S&P 500 index. Financial markets often exhibit sharp moves (jumps) that simple diffusion models such as Black-Scholes cannot capture. Our model aims to capture both the continuous volatility and these abrupt jumps.

The workflow is:
1.  **Data download and preparation**: download the historical S&P 500 series (`^GSPC`) and compute daily log returns.
2.  **Parameter estimation**: fit our jump-diffusion model to the historical data to estimate its key parameters (`μ`, `σ`, jump probability, etc.).
3.  **Analysis of results**: interpret the estimated parameters and assess the quality of the fit.
4.  **Simulation and comparison**: use the estimated parameters to simulate a new return series and compare its distribution with the real data to validate the model.
5.  **Comparing jump distributions**: try the five jump distributions available in the library (Normal, Skew-Normal, SGED, Kou, Student-t) and see which one best describes the data.

> Hay una versión en español de este notebook: `sp500_jump_diffusion_example.ipynb`.

## 1. Environment setup and data loading

In [ ]:
# Install the jump-diffusion-estimation library from PyPI
%pip install -q jump-diffusion-estimation

In [ ]:
%pip install yfinance matplotlib seaborn -q

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from jump_diffusion.estimation import JumpDiffusionEstimator
from jump_diffusion.simulation import JumpDiffusionSimulator

# Plot styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

### Downloading the S&P 500 data
We use the `yfinance` library to download the adjusted close prices of the S&P 500 index (`^GSPC`) from 2015 to early 2024.

In [ ]:
# 🔧 Try changing this -- other interesting tickers: 'AAPL', '^IXIC' (Nasdaq),
# 'BTC-USD', '^MXX' (Mexico IPC), 'EURUSD=X', 'TSLA'...
TICKER = '^GSPC'
START = '2015-01-01'
END = '2024-01-01'

In [ ]:
data = yf.download(TICKER, start=START, end=END, progress=False)

# Depending on the installed yfinance version, 'Close' may come as a
# MultiIndex column (ticker, field) or as a flat Series. Handle both.
close = data['Close']
prices = (close[TICKER] if isinstance(close, pd.DataFrame) else close).dropna()

print(f"First 5 prices of {TICKER}:")
print(prices.head())

Let's look at the original series:

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(prices.index, prices.values, alpha=0.8, color='b')
plt.title(f'Closing value series of {TICKER}')
plt.xlabel('Date')
plt.ylabel('Value')
plt.grid(True)
plt.show()

### Computing log returns
Financial models usually work with **log returns** (`log(P_t) - log(P_{t-1})`), which have more desirable statistical properties (such as stationarity). These returns are the "increments" in our model.

In [ ]:
log_returns = np.diff(np.log(prices.values))

# The time step (dt) is daily. We assume 252 trading days per year.
dt = 1/252

print(f"Computed {len(log_returns)} daily log returns.")

### Visualizing the returns
A first look at the return series lets us observe the volatility and the extreme spikes (possible jumps), such as those during the COVID-19 crisis in 2020.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(prices.index[1:], log_returns, alpha=0.8, color='b')
plt.title(f'Daily log returns of {TICKER}')
plt.xlabel('Date')
plt.ylabel('Log return')
plt.grid(True)
plt.show()

Distribution of the increments

In [ ]:
plt.figure(figsize=(14, 7))
plt.hist(log_returns, color='b', bins=100)
plt.title(f'Daily log returns of {TICKER}')
plt.xlabel('Increments')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

## 2. Estimating the jump-diffusion model

In [ ]:
# Initialize the estimator with the returns and dt
estimator = JumpDiffusionEstimator(log_returns, dt)

# Fit by maximum likelihood
results = estimator.estimate()

if not results['convergence']:
    print("\u26A0\uFE0F  The optimizer did not converge; the estimated parameters may be unreliable.")

# Show a diagnostic report of the results
estimator.diagnostics()

## 3. Simulation and distribution comparison

Now the crucial step: how well does our model capture reality? To answer this, we simulate a jump-diffusion process using the parameters we just estimated. Then we compare the distribution of the simulated returns with the distribution of the real S&P 500 returns.

In [ ]:
# Extract the estimated parameters
estimated_params = results['parameters']

# Build a simulator with these parameters
simulator = JumpDiffusionSimulator(**estimated_params)

# Simulate a path with the same number of steps as the original data
T = len(log_returns) * dt
n_steps = len(log_returns)

# The model was calibrated on log returns, so its state variable
# represents log-price: x0 must be log(P0), not the raw price.
_, simulated_path, _ = simulator.simulate_path(
    T=T, n_steps=n_steps, x0=np.log(prices.iloc[0]), seed=42
)

# simulated_path is already on a log-price scale, so its increments
# are directly the simulated log returns (no second np.log).
simulated_log_returns = np.diff(simulated_path)

Let's now look at the behavior of the simulated data.

In [ ]:
simulator.plot_simulation()

### Graphical comparison of the distributions

We visualize both distributions (real and simulated) via histograms and kernel density estimates (KDE). A good fit shows a large overlap between the two curves.

In [ ]:
plt.figure(figsize=(14, 6))

# Calculate common x-axis limits
all_returns = np.concatenate((log_returns, simulated_log_returns))
x_min = all_returns.min()
x_max = all_returns.max()

plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
hist_real, bins_real, _ = plt.hist(log_returns, bins=100, alpha=0.8, color='royalblue', density = True)
plt.title(f'{TICKER} Log Returns (Real)')
plt.xlabel('Log Returns')
plt.ylabel('Frequency')
plt.grid(True)
plt.xlim(x_min, x_max) # Set common x-axis

plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
hist_sim, bins_sim, _ = plt.hist(simulated_log_returns, bins=100, alpha=0.8, color='orangered', density = True)
plt.title('Simulated Log Returns')
plt.xlabel('Log Returns')
plt.ylabel('Frequency')
plt.grid(True)
plt.xlim(x_min, x_max) # Set common x-axis

# Calculate common y-axis limits after plotting both histograms
y_max = max(hist_real.max(), hist_sim.max()) * 1.05 # Add a small buffer

plt.subplot(1, 2, 1) # Set ylim for the first subplot
plt.ylim(0, y_max)
plt.subplot(1, 2, 2) # Set ylim for the second subplot
plt.ylim(0, y_max)

plt.tight_layout() # Adjust layout to prevent overlapping
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))

# Histogram and KDE for the real S&P 500 data
sns.histplot(log_returns, bins=100, kde=True, stat='density', color='royalblue', label=f'{TICKER} (Real)')

# Histogram and KDE for the simulated data
sns.histplot(simulated_log_returns, bins=100, kde=True, stat='density', color='orangered', alpha=0.7, label='Simulated (with estimated parameters)')

plt.title('Return distribution comparison: Real vs. Simulated', fontsize=16)
plt.xlabel('Log returns', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.show()

### Conclusion

The comparison plot shows that the distribution of the simulated returns closely resembles that of the real data. In particular, the model seems to capture well:

-   The **central peak** around zero, representing low-volatility days.
-   The **heavier tails** (leptokurtosis) than a normal model, indicating that extreme events are more likely than a Gaussian distribution would suggest.

That said, the real S&P 500 exhibits even greater kurtosis than the model: its central peak is taller and narrower than the simulated one, suggesting that the real data concentrate an even larger share of very-low-volatility days than our diffusion + skew-normal-jump mixture reproduces. This is reasonable -- a single Bernoulli jump regime is a simplification of the real dynamics, where volatility likely varies over time (volatility clustering) rather than being constant between jumps.

Overall the fit is solid: the jump-diffusion model captures both the central peak and the heavy tails substantially better than a simple (Gaussian) diffusion model, validating it as a reasonable tool for describing the dynamics of S&P 500 returns.

## 4. Comparing jump distributions

So far we used the default skew-normal distribution. But the library lets you plug in different jump distributions (`jump_distribution=`) without changing anything else in the estimation flow. Let's fit the five available ones to the same returns and compare them by AIC/BIC and a parametric-bootstrap Kolmogorov-Smirnov test, using `JumpDistributionComparison`.

In [ ]:
from jump_diffusion.distributions import (
    KouJump,
    NormalJump,
    SGEDJump,
    SkewNormalJump,
    StudentTJump,
)
from jump_diffusion.validation import JumpDistributionComparison

comparison = JumpDistributionComparison(log_returns, dt)
comparison.fit("Normal", NormalJump(), seed=42)
comparison.fit("SkewNormal", SkewNormalJump(), seed=42)
comparison.fit("SGED", SGEDJump(), seed=42)
comparison.fit("Kou", KouJump(), seed=42)
comparison.fit("StudentT", StudentTJump(), seed=42)

comparison.compare()

In our reference run (`^GSPC`, 2015-2024), the AIC ranking was:

1. **Student-t** (AIC ≈ -14550) -- the best, and the only one whose KS p-value (≈0.79) does not reject that the simulated returns come from the same distribution as the real ones.
2. **Kou** (AIC ≈ -14522), also reasonable (KS p-value ≈ 0.36).
3. **Skew-Normal** and **Normal**, well behind, with KS p-values < 0.01 -- the test rejects that they capture the real distribution well.
4. **SGED** -- ⚠️ in this run the optimizer (`L-BFGS-B`) **did not converge** from its default initial guess (`convergence: False`). This is a known limitation of the local optimizer against the SGED mixture likelihood (the thesis's own applied finding, comparing BFGS/L-BFGS-B against Differential Evolution) -- not necessarily a limitation of the distribution itself. It is exactly why the library also provides **Differential Evolution** as an alternative estimation method (`method="differential_evolution"`), which is far more robust here.

If you change `TICKER` in Section 1 and rerun the notebook, the ranking may well change!

In [ ]:
comparison.plot_comparison()

---
➡️ Want to play with these same distributions interactively (sliders, free simulation, and even a guess-the-parameters game)? Try [`jump_diffusion_playground_en.ipynb`](https://colab.research.google.com/github/jdospina/jump-diffusion-estimation/blob/main/notebooks/jump_diffusion_playground_en.ipynb) (opens in Colab).

## Standard errors via likelihood profiling

Once the parameters are estimated (preferably using Differential Evolution to secure the global optimum), we can compute standard errors and 95% confidence intervals using **likelihood profiling**.

This technique is far more robust than numerically computing the Hessian matrix (which is often unstable for complex mixtures).

In [ ]:
# Estimate standard errors by profiling (n_points=5 for speed here).
# Make sure you have run estimate() on the estimator first.
try:
    # Try estimator_de if it exists (from the Differential Evolution showcase)
    est = estimator_de
except NameError:
    try:
        # Try estimator if it exists (from the general example)
        est = estimator
    except NameError:
        print("Make sure to define an 'estimator' and run estimate() first.")
        est = None

if est is not None:
    print("Computing standard errors...")
    se_results = est.estimate_standard_errors(n_points=5, confidence_level=0.95)

    # Print the updated diagnostics, now including the SE / interval table
    est.diagnostics()

    # Plot the profile likelihood curves for each parameter
    est.plot_profiles()